In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# =====================================================
# 1. Wczytanie danych
# =====================================================
df = pd.read_csv("climate_deforestation_supervised_modeling_base.csv")

# =====================================================
# 2. Definicja klastrów
# =====================================================
cluster_map = {
    "AC": 0, "AM": 0, "AP": 0, "MA": 0, "PA": 0, "RO": 0, "TO": 0,

    "PR": 1, "RS": 1, "SC": 1, "SP": 1,

    "DF": 2, "ES": 2, "GO": 2, "MG": 2, "MS": 2,

    "AL": 3, "BA": 3, "CE": 3, "MT": 3, "PB": 3, "PE": 3,
    "RN": 3, "RR": 3, "SE": 3
}

df["Cluster"] = df["State"].map(cluster_map)

# usunięcie stanów spoza klastrów, jeśli istnieją
df = df.dropna(subset=["Cluster"]).copy()
df["Cluster"] = df["Cluster"].astype(int)

# =====================================================
# 3. Zmienne sezonowe
# Brazylia: lato = grudzień, styczeń, luty
#          zima = czerwiec, lipiec, sierpień
# =====================================================
summer = (
    df[df["Month"].isin([12, 1, 2])]
    .groupby(["State", "Year"])["Air_Temperature"]
    .mean()
    .reset_index()
    .rename(columns={"Air_Temperature": "Summer_Temperature"})
)

winter = (
    df[df["Month"].isin([6, 7, 8])]
    .groupby(["State", "Year"])["Air_Temperature"]
    .mean()
    .reset_index()
    .rename(columns={"Air_Temperature": "Winter_Temperature"})
)

season_df = pd.merge(
    summer,
    winter,
    on=["State", "Year"],
    how="inner"
)

season_df["Annual_Seasonal_Temperature"] = (
    season_df["Summer_Temperature"] +
    season_df["Winter_Temperature"]
) / 2

season_df["Seasonal_Temperature_Range"] = (
    season_df["Summer_Temperature"] -
    season_df["Winter_Temperature"]
).abs()

season_df["Summer_Deviation"] = (
    season_df["Summer_Temperature"] -
    season_df["Annual_Seasonal_Temperature"]
)

season_df["Winter_Deviation"] = (
    season_df["Winter_Temperature"] -
    season_df["Annual_Seasonal_Temperature"]
)

# =====================================================
# 4. Agregacja miesięcy -> lata
# =====================================================
annual = (
    df.groupby(["State", "Year", "Cluster"])
      .agg(
          Annual_Deforestation=("Deforestation_ha", "sum"),

          Mean_Temperature=("Air_Temperature", "mean"),
          Max_Temperature=("Max_Air_Temperature", "mean"),
          Min_Temperature=("Min_Air_Temperature", "mean"),

          Mean_Precipitation=("Total_Precipitation", "mean"),
          Mean_Humidity=("Relative_Humidity", "mean"),
          Max_Humidity=("Max_Relative_Humidity", "mean"),
          Min_Humidity=("Min_Relative_Humidity", "mean"),

          Mean_Radiation=("Global_Radiation", "mean"),
          Mean_Pressure=("Atmospheric_Pressure_Station", "mean")
      )
      .reset_index()
)

# =====================================================
# 5. Dołączenie sezonowości
# =====================================================
annual = annual.merge(
    season_df,
    on=["State", "Year"],
    how="left"
)

# =====================================================
# 6. Dodatkowe zmienne klimatyczne
# =====================================================
annual["Temperature_Range"] = (
    annual["Max_Temperature"] -
    annual["Min_Temperature"]
)

annual["Humidity_Range"] = (
    annual["Max_Humidity"] -
    annual["Min_Humidity"]
)

# =====================================================
# 7. Target: czy wylesianie było ponadprzeciętne
# =====================================================
threshold = annual["Annual_Deforestation"].median()

annual["High_Deforestation"] = (
    annual["Annual_Deforestation"] > threshold
).astype(int)

print("="*60)
print("Próg ponadprzeciętnego wylesiania:")
print(threshold)
print("="*60)

print("\nRozkład klas:")
print(annual["High_Deforestation"].value_counts())

# =====================================================
# 8. Features
# używamy danych klimatycznych + klastrów
# NIE używamy Annual_Deforestation jako feature,
# bo to byłby data leakage
# =====================================================
features = [
    "Year",
    "Cluster",

    "Mean_Temperature",
    "Max_Temperature",
    "Min_Temperature",
    "Temperature_Range",

    "Mean_Precipitation",
    "Mean_Humidity",
    "Max_Humidity",
    "Min_Humidity",
    "Humidity_Range",

    "Mean_Radiation",
    "Mean_Pressure",

    "Summer_Temperature",
    "Winter_Temperature",
    "Annual_Seasonal_Temperature",
    "Seasonal_Temperature_Range",
    "Summer_Deviation",
    "Winter_Deviation"
]

target = "High_Deforestation"

# usunięcie braków
model_df = annual.dropna(subset=features + [target]).copy()

X = model_df[features]
y = model_df[target]

# =====================================================
# 9. Podział train/test
# =====================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# =====================================================
# 10. Modele
# =====================================================
models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

# =====================================================
# 11. Trenowanie i ewaluacja modeli
# =====================================================
results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline.named_steps["model"], "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = np.nan

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    })

    print("\n" + "="*60)
    print(name)
    print("="*60)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

# =====================================================
# 12. Tabela wyników
# =====================================================
results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("PORÓWNANIE MODELI")
print("="*60)
print(results_df)

# =====================================================
# 13. Cross-validation
# =====================================================
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=cv,
        scoring="f1"
    )

    cv_results.append({
        "Model": name,
        "CV_F1_mean": scores.mean(),
        "CV_F1_std": scores.std()
    })

cv_results_df = pd.DataFrame(cv_results)

print("\n" + "="*60)
print("CROSS-VALIDATION F1")
print("="*60)
print(cv_results_df)

# =====================================================
# 14. Feature importance dla Random Forest
# =====================================================
rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_model = rf_pipeline.named_steps["model"]

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\n" + "="*60)
print("WAŻNOŚĆ CECH - RANDOM FOREST")
print("="*60)
print(importance_df)

# =====================================================
# 15. Predykcje Random Forest
# =====================================================
rf_pred = rf_pipeline.predict(X_test)
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]

prediction_df = X_test.copy()
prediction_df["Actual_High_Deforestation"] = y_test.values
prediction_df["Predicted_High_Deforestation"] = rf_pred
prediction_df["Probability_High_Deforestation"] = rf_proba

prediction_df = prediction_df.merge(
    model_df[["State", "Year", "Annual_Deforestation", "High_Deforestation"]],
    left_index=True,
    right_index=True,
    how="left"
)

# =====================================================
# 16. Zapis wyników
# =====================================================
annual.to_csv(
    "annual_climate_deforestation_classification_dataset.csv",
    index=False
)

results_df.to_csv(
    "classification_model_results.csv",
    index=False
)

cv_results_df.to_csv(
    "classification_cross_validation_results.csv",
    index=False
)

importance_df.to_csv(
    "random_forest_classification_feature_importance.csv",
    index=False
)

prediction_df.to_csv(
    "random_forest_classification_predictions.csv",
    index=False
)

print("\n" + "="*60)
print("ZAPISANO PLIKI")
print("="*60)
print("- annual_climate_deforestation_classification_dataset.csv")
print("- classification_model_results.csv")
print("- classification_cross_validation_results.csv")
print("- random_forest_classification_feature_importance.csv")
print("- random_forest_classification_predictions.csv")

Próg ponadprzeciętnego wylesiania:
46078.0

Rozkład klas:
High_Deforestation
0    246
1    246
Name: count, dtype: int64

Decision Tree

Confusion Matrix:
[[48 12]
 [17 45]]

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.80      0.77        60
           1       0.79      0.73      0.76        62

    accuracy                           0.76       122
   macro avg       0.76      0.76      0.76       122
weighted avg       0.76      0.76      0.76       122


Random Forest

Confusion Matrix:
[[57  3]
 [13 49]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.95      0.88        60
           1       0.94      0.79      0.86        62

    accuracy                           0.87       122
   macro avg       0.88      0.87      0.87       122
weighted avg       0.88      0.87      0.87       122


Gradient Boosting

Confusion Matrix:
[[56  4]
 [13 49]]

Classificatio